# LAB GenAI - LLMs - OpenAI GPT API Exercises

## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [9]:
import os
import json
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import tiktoken

pd.set_option("display.max_colwidth", 120)

load_dotenv()                                   # reads OPENAI_API_KEY from the .env file
print("Key found:", "OPENAI_API_KEY" in os.environ)   # never print the key itself

client = OpenAI()                               # finds the key in the environment
MODEL = "gpt-5.4-nano"                          # the cheapest current model; change it in one place

total_usage = {"input": 0, "output": 0}

def chat(messages, **options):
    """Send messages to the model, keep count of the tokens, and return the text of the reply."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    response = client.chat.completions.create(model=MODEL, messages=messages, **options)
    total_usage["input"] += response.usage.prompt_tokens
    total_usage["output"] += response.usage.completion_tokens
    return response.choices[0].message.content

Key found: True


In [16]:
history = [{"role": "system", "content": "You are the support assistant of a bank. Answer in at most two sentences."}]

def say(text):
    history.append({"role": "user", "content": text})
    answer = chat(history)
    history.append({"role": "assistant", "content": answer})
    return answer

print(say("Hi! My name is Lucía and my card has not arrived."))
print("---")
print(say("What is my name, and what was my problem? Be funny!"))


Hi Lucía—sorry about that. Please share the card type (debit/credit), where it was sent to, and the order/reference date so we can check the delivery status and next steps.
---
Your name is Lucía, and your problem is that your card hasn’t arrived yet—basically your plastic is on a very slow “grand adventure.” If you tell me the card type and delivery address, I’ll help speed up the quest.


In [12]:
for temperature in [0, 1.5]:
    response = client.chat.completions.create(
        model=MODEL, n=3, temperature=temperature,
        messages=[{"role": "user", "content": "Suggest a name for a savings app. Answer with the name only."}])
    print(f"temperature={temperature}:", [c.message.content for c in response.choices])

temperature=0: ['NestEgg', 'NestEgg', 'NestEgg']
temperature=1.5: ['NestEgg', 'SproutSave', 'MintMinder']


In [13]:
response = client.chat.completions.create(
    model=MODEL, max_completion_tokens=15,
    messages=[{"role": "user", "content": "Explain how a credit card works."}])
print(response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)


A credit card lets you borrow money from the card issuer (
Finish reason: length


In [ ]:
question = "Give me three tips for saving money."

print("\nTop_p:")
print(chat(question, top_p=0.3))

print("\nFrequency penalty:")
print(chat(question, frequency_penalty=1.0)) #Positive values discourage repeated words

print("\nPresence penalty:")
print(chat(question, presence_penalty=1.0)) #Positive values encourage introducing new concepts

# print("\nStop:")
# print(chat(question, stop=["2."]))


Top_p:
1. **Set a budget and track spending**: Pick a monthly limit for categories (food, transport, fun) and check your spending weekly so you catch overspending early.  
2. **Cut one recurring expense**: Review subscriptions, memberships, and bills—cancel or downgrade one thing you don’t use often (or negotiate a lower rate).  
3. **Use a “cooling-off” rule for purchases**: Wait 24–48 hours before buying non-essential items to avoid impulse buys, and compare prices (or use a list) when you do buy.

Frequency penalty:
1. **Make a budget and track spending** — Pick a monthly limit for essentials (rent, food, bills) and set categories for other spending so you know where your money is going.  
2. **Cut or reduce one “easy” expense** — Look for subscriptions, fees, or recurring services you don’t fully use; cancel or downgrade at least one.  
3. **Use a savings habit automatically** — Set up an automatic transfer to savings right after payday (even small amounts), and try to avoid touch

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [17]:
text = """Grading homework and providing feedback to students is arguably the toughest part of being an educator. During my tenure as a professor, reviewing the results of carefully crafted assignments was at best a barometer of how well I was relaying the concepts and crucial elements of the course. In the era of AI, it’s become an infuriating exercise in futility. 
Discussion posts littered with complex references to materials outside the course’s accepted reading. Students who were instant masters at the subject matter from Day 1, taking a page from the classic tactic of “forge your mom’s signature the first time so you can keep forging it forever.” 
After reading a story last week about a UC Berkeley math professor who no longer bothers grading his students’ homework because they’re all using AI, I couldn’t help reflecting on my own experience as a professor of practice at the Walter Cronkite School of Journalism and Mass Communication at Arizona State University. With the proliferation of AI tools once ChatGPT launched in late 2022, and tech giants worldwide started following suit, launching increasingly sophisticated chatbots and now AI agents, grading homework became deeply depressing. 
More from CNET

In advanced communications courses like the ones I taught, reviewing homework turned into a hypervigilant hunt for the hallmarks of AI-generated text and ideas. The beast was cunning, quick and never slept. My tools for catching it were sorely lacking.

CNET AI Atlas badge; click to see more
Despite there being no way to verify it, I was fairly certain there were online students in my classes who had never done any of the work themselves the entire course. When those students were given work they could not outsource to AI, like simulations and recorded oral presentations, they reliably struggled to maintain the perfect standard they had set. But when you get A’s all semester, one C or D doesn’t really matter. The AI cheating math still shakes out.

In an effort to keep myself alert to the use of AI tools in completing homework, I always ran a smattering of assignments through a few widely available AI chat tools to compare against student-submitted work. 

Most of the time, this method was inconclusive, but by the time I left teaching earlier this year, I suspected that at least 90% of the students I had online were using AI to do all or part of their work. 

In my very small in-person classes, I had several confrontations over the unauthorized, clandestine use of AI to complete assignments. One in particular made me rethink the purpose of my work.

A student gave a presentation early in the semester, and when asked a few basic questions in front of the class about how they came up with their ideas, the student was flummoxed, unable to explain the circumstances that led to their own creative output. 

We had a talk privately later that day about AI use, and the student admitted to using ChatGPT to create part of their assignment. They agreed to never use it again in my class. On the next assignment, we ended up having the same discussion, and I was given the same promise. It didn’t stick. 

When I went to my supervising dean about the use of AI in my classroom and inquired as to whether this case should be referred to the academic integrity board, they told me it was at my discretion and that, candidly, the university and college were grappling with what stance to take in the face of widespread AI adoption. ASU’s stance on AI currently reads, in part: “Our focus is on using AI not only to address educational challenges, but to do so in a way that respects and uplifts all learners.”

Education is about the journey, not the outcome
Some might argue that cheating is a natural impulse. On your taxes, on the Monopoly board. Sometimes putting a thumb on the scale is the only way we feel we can win at a game that looks rigged or too hard to beat. 

But education isn’t a game. Even though school environments can be competitive, learning is not something at which you “win.” It’s a personal journey meant to unlock fluency you can call on at will, like knowing how to speak a language or effortlessly opening a door. 

AI may be viewed by today’s students as a skeleton key, granting instant access to great grades and passable answers that a busy, bored teacher can’t flag as 100% fake. Students who outsource the work of learning take for granted the value of being able to demonstrate expertise with ease and risk ruining their credibility at their own peril.

Until our brains are plugged into our computers 24/7, it’s a crutch that educators can’t afford to quibble about any longer."""

messages = [
    {"role": "system", "content": "Summarize the following text in at most three sentences."},
    {"role": "user", "content": text}
]

print("Summary:")
print(chat(messages, temperature=0, max_completion_tokens=150))


Summary:
The author describes how AI has made grading homework and giving meaningful feedback increasingly futile, pushing them into trying to detect AI-generated work with unreliable tools. They recount suspecting widespread AI use, confronting students who admitted using ChatGPT, and struggling with unclear institutional policies while trying to uphold academic integrity. Ultimately, they argue that education is a personal journey and that outsourcing learning to AI undermines credibility and fluency.


In [18]:
for temperature in [0, 1.5]:
    print(f"\ntemperature={temperature}:")
    print(chat(messages, temperature=temperature))


temperature=0:
The author describes how grading homework became increasingly demoralizing after AI tools like ChatGPT made it hard to tell whether students actually learned the material. They recount suspicions and confrontations with students using AI, including a case where a student couldn’t explain their own ideas and repeatedly promised not to use AI again. Ultimately, they argue that education is a personal journey, and relying on AI as a “crutch” undermines genuine learning and credibility.

temperature=1.5:
The author describes how, after the rise of AI tools, grading homework became nearly impossible and demoralizing, leading to heavy suspicion that many students outsourced assignments. They recount trying to detect AI use with limited methods, and a case where a student admitted to using ChatGPT and later repeatedly made the same promises without following through. Ultimately, the piece argues that education is a learning journey, and using AI to bypass work undermines genui

In [19]:
response = client.chat.completions.create(
    model=MODEL, max_completion_tokens=20,
    messages=messages
)
print("\nShort summary:", response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)


Short summary: The author describes how AI has made grading homework and giving meaningful feedback feel futile, as
Finish reason: length


In [20]:
print("\nTop_p:")
print(chat(messages, top_p=0.3))


Top_p:
The author describes how grading homework became increasingly demoralizing after AI tools spread, making it hard to tell whether students truly learned the material or outsourced work. They recount suspicions that most online students used AI and a few in-person confrontations where students admitted using ChatGPT but repeatedly failed to stop. Ultimately, they argue that education is a personal journey, and relying on AI as a “crutch” undermines genuine learning and credibility.


In [21]:
import math

response = client.chat.completions.create(
    model=MODEL, logprobs=True, top_logprobs=3,
    messages=messages
)

first = response.choices[0].logprobs.content[0]

print("\nSummary:", response.choices[0].message.content)
print([(t.token, round(math.exp(t.logprob), 3)) for t in first.top_logprobs])



Summary: The author describes how, since AI tools became widely available after ChatGPT launched in 2022, grading homework has become increasingly futile and frustrating due to likely AI-assisted cheating that’s hard to detect. They recount attempts to verify student work with AI comparisons, confrontations in small classes, and an incident where a student couldn’t explain their own presentation, despite promises to stop using AI. Ultimately, they argue that learning is a personal journey rather than a game, and that outsourcing learning with AI undermines genuine expertise and credibility.
[('The', 0.99)]


## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [22]:
text = "My card has not arrived after nine days."

# Translation using the chat() function from the lecture
messages = [
    {"role": "system", "content": "You are a professional translator. Translate the text into German. Answer with the translation only."},
    {"role": "user", "content": text}
]

print("Original:", text)
print("Translation:", chat(messages, temperature=0, max_completion_tokens=100))


Original: My card has not arrived after nine days.
Translation: Meine Karte ist nach neun Tagen noch nicht angekommen.


In [24]:
for temperature in [0, 2]:
    print(f"\ntemperature={temperature}:")
    print(chat(messages, temperature=temperature))


temperature=0:
Meine Karte ist nach neun Tagen noch nicht angekommen.

temperature=2:
Meine Karte ist auch nach neun Tagen nicht angekommen.


## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [25]:
reviews = [
    "Loved it!",
    "The package arrived on Tuesday.",
    "Not worth the money, it broke in a week."
]

print("Zero-shot classification:")

for review in reviews:
    print(chat(f"What is the sentiment of this review? {review}"))


few_shot = [
    {"role": "system", "content": "Classify the sentiment of the review as positive, negative or neutral."},
    {"role": "user", "content": "The hotel was clean and the staff were lovely."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "The package arrived on Monday."},
    {"role": "assistant", "content": "neutral"},
    {"role": "user", "content": "The app crashes every time I open it."},
    {"role": "assistant", "content": "negative"},
]
print("\nFew-shot classification:")

for review in reviews:
    print(f"{chat(few_shot + [{'role': 'user', 'content': review}], temperature=0):<9} {review}")

Zero-shot classification:
The sentiment of the review **“Loved it!”** is **positive**.
I can’t determine the sentiment from that review text alone. “The package arrived on Tuesday” is a neutral, factual statement (no clear praise or complaint).
The sentiment of the review is **negative**.

Few-shot classification:
positive  Loved it!
neutral   The package arrived on Tuesday.
negative  Not worth the money, it broke in a week.


## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [26]:
prompt = "One evening, a scientist discovered a mysterious machine in the laboratory."

messages = [
    {"role": "system", "content": "You are a creative writer. Continue the story in one short paragraph."},
    {"role": "user", "content": prompt}
]

print("Initial text:", prompt)
print("\nCompletion:")
print(chat(messages, temperature=1.5, max_completion_tokens=200))


for temperature in [0, 1.5]:
    print(f"\ntemperature={temperature}:")
    print(chat(messages, temperature=temperature))

Initial text: One evening, a scientist discovered a mysterious machine in the laboratory.

Completion:
The machine was neither bolted down nor wired to anything sensible; it just sat amid dust and humming transformers, its brass rings engraved with symbols that seemed to shift when he blinked. Dr. Halden, convinced it had to be a prank until the lights dimmed by themselves, pressed a gloved palm to the cold metal. The chamber swallowed sound, then answered with a soft, impossible chime—as if somewhere else a clock had just remembered how to count. A thin ribbon of light spilled from the machine’s core, drawing patterns in the air like handwriting, and a familiar scent—stormwater and orange peel—rose from nowhere. Halden leaned closer, heart thudding, when the ribbon rearranged into a warning only he would understand: *DO NOT TURN IT BACK*. Before he could steady his breath, the laboratory windows frost over in sudden spirals, and the machine began to speak in pulses through the floorbo

# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)